In [ ]:
# Path configuration
from pathlib import Path

PROJECT_ROOT = Path("..")  # Repository root.
BASELINE = PROJECT_ROOT / "data"
FIRM_DATA = PROJECT_ROOT
PATSTAT = PROJECT_ROOT / "patstat"
PATENT_TEXT = PROJECT_ROOT / "patent_text"
SAMPLEDATA = PROJECT_ROOT / "sampledata"
DATA = Path("data")          # Generated revision intermediates.
OUTPUTS = Path("outputs")    # Tables, figures and saved comparison results.
SEED = 42


# Alternative GBB identification methods

Calculate the SI S4 comparisons. First, fit HDBSCAN and k-means on the same five-dimensional UMAP representation of the detailed CPC profile. Then construct continuous GBBs from orthogonal residuals of matched-family vectors and compare them with the discrete GBBs.

Set paths in the first cell. `KMEANS_PROFILE` stores the fitted code profile and labels; `CONTINUOUS_PROFILE` stores the positive centroid–CPC similarities. Each comparison reads its saved profile, so fitting and comparison can be run separately.


In [ ]:
import numpy as np
import pandas as pd
from gbb_utils import (nonnoise, rca, correspondence_metrics, nmi_permutations,
                       reduce_profile, cluster_profile)

WORK = DATA / "alternatives"
WORK.mkdir(parents=True, exist_ok=True)
KMEANS_PROFILE = WORK / "hdbscan_kmeans.parquet"
CONTINUOUS_WORK = WORK / "continuous"
CONTINUOUS_PROFILE = BASELINE / "clus_cpc.parquet"
out = OUTPUTS / "alternatives"
out.mkdir(parents=True, exist_ok=True)


## Detailed-CPC k-means fitting calculation

In [ ]:
def fit_kmeans(profile, umap_coordinates):
    """Fit 82 k-means centers with code-frequency weights."""
    from sklearn.cluster import MiniBatchKMeans

    model = MiniBatchKMeans(n_clusters=82, random_state=0, batch_size=10, n_init="auto")
    model.fit(umap_coordinates, sample_weight=profile["count"])
    result = profile.copy()
    result["clus2"] = ["c" + str(label) for label in model.labels_]
    return result


In [ ]:
from gensim.models import Word2Vec

profile = pd.read_parquet(BASELINE / "clusangle_outlier_hdbscan2.parquet")
vectors = Word2Vec.load(str(BASELINE / "external/cpc_sg_ns_dim50.model")).wv
coordinates = reduce_profile(profile, vectors, neighbors=15, dimensions=5, seed=SEED)
# Both methods use the same coordinates; only k-means uses frequency weights.
comparison = fit_kmeans(cluster_profile(profile, coordinates, min_size=100), coordinates)
comparison.to_parquet(KMEANS_PROFILE, index=False)
comparison.head()


## Compare the saved k-means and HDBSCAN labels

Request 82 k-means centers and weight codes by frequency when fitting. Weight shared codes by squared mean strength for numerical NMI, and by code counts for the correspondence figure.


In [ ]:
# Read the comparison fit saved above; retain its shared nonnoise CPC codes.
comparison = pd.read_parquet(KMEANS_PROFILE)
shared = nonnoise(comparison).sort_values("cpc").copy()
shared["score"] = shared["mean"] ** 2
summary, null = nmi_permutations(shared, "clus", "clus2", "score", seed=20260915)
pd.DataFrame([dict(comparison="kmeans", **summary)]).to_csv(out / "kmeans_nmi.csv", index=False)
pd.DataFrame({"nmi": null}).to_csv(out / "kmeans_nmi_null.csv", index=False)
weighted = rca(shared, "clus", "clus2", "score")
weighted.to_csv(out / "kmeans_weighted_correspondence.csv", index=False)
# Plot code counts and calculate numerical NMI with squared-strength weights.
counts = shared.groupby(["clus", "clus2"]).size().rename("count").reset_index()
rca(counts, "clus", "clus2", "count").to_csv(out / "kmeans_count_correspondence.csv", index=False)


## Continuous orthogonal extraction

Sum CPC vectors within each family, remove the nongreen projection from each green vector, cluster distinct residuals, and average pair residuals within clusters. Save the positive centroid–CPC profile to `CONTINUOUS_PROFILE`. Save residual arrays, labels and centroids under `CONTINUOUS_WORK`.


In [ ]:
import scipy.sparse as sp
from gensim.models import Word2Vec


CONTINUOUS_WORK.mkdir(parents=True, exist_ok=True)
MODEL_PATH = BASELINE / 'external/cpc_sg_ns_dim50.model'
PAIR_PATH = BASELINE / 'matched_ccmt_other.parquet'
BATCH_SIZE = 100_000
N_NEIGHBORS = 15
N_COMPONENTS = 5
MIN_SAMPLES = 50
MIN_CLUSTER_SIZE = 5000


### Matched families and CPC assignments

Both patent-ID columns in the matching table contain DOCDB family IDs. Keep matched pairs in table order. Exclude Y02 codes from family vector sums; each distinct family/code assignment contributes its raw CPC vector once.


In [ ]:
pairs = pd.read_parquet(PAIR_PATH, columns=['ccmt_patent_id', 'patent_id', 'cos'])
families = pd.Index(np.union1d(pairs.ccmt_patent_id, pairs.patent_id), name='docdb_family_id')
cpc2vec = Word2Vec.load(str(MODEL_PATH)).wv

cpc = pd.read_parquet(PATSTAT / 'tls225.parquet',
                      columns=['docdb_family_id', 'cpc_class_symbol'])
cpc = cpc[cpc.docdb_family_id.isin(families)].copy()
cpc['cpc'] = cpc.cpc_class_symbol.str.replace(' ', '', regex=False)
cpc = cpc.loc[~cpc.cpc.str.startswith('Y02') & cpc.cpc.isin(cpc2vec.key_to_index),
              ['docdb_family_id', 'cpc']].drop_duplicates()
codes = pd.Index(sorted(cpc.cpc.unique()), name='cpc')

incidence = sp.coo_matrix(
    (np.ones(len(cpc), dtype=np.int64),
     (families.get_indexer(cpc.docdb_family_id), codes.get_indexer(cpc.cpc))),
    shape=(len(families), len(codes)),
).tocsr()
family_vectors = incidence @ cpc2vec[codes.tolist()]
pd.Series({'matched_pairs': len(pairs), 'families': len(families),
           'vector_codes': len(codes), 'family_code_incidences': incidence.nnz})


### Orthogonal residual for each matched pair

For green vector `g` and nongreen vector `n`, calculate `g - (g·n)/(n·n + 1e-6) * n`. Write the pair residuals in batches to a NumPy memory-mapped array.


In [ ]:
green_rows = families.get_indexer(pairs.ccmt_patent_id)
nongreen_rows = families.get_indexer(pairs.patent_id)
residuals = np.lib.format.open_memmap(
    CONTINUOUS_WORK / 'orthemb.npy', mode='w+', dtype=family_vectors.dtype,
    shape=(len(pairs), family_vectors.shape[1]),
)
for start in range(0, len(pairs), BATCH_SIZE):
    stop = min(start + BATCH_SIZE, len(pairs))
    g = family_vectors[green_rows[start:stop]]
    n = family_vectors[nongreen_rows[start:stop]]
    coefficient = np.einsum('ij,ij->i', g, n) / (np.einsum('ij,ij->i', n, n) + 1e-6)
    residuals[start:stop] = g - coefficient[:, None] * n
residuals.flush()
del family_vectors, incidence
residuals.shape


### Distinct residuals, cosine neighbours and five-dimensional UMAP

Cluster each distinct residual once. Save an inverse index mapping pair rows to distinct residuals for the centroid calculation. Build a cosine-neighbour graph and pass it to UMAP through `precomputed_knn`, using `SEED` for the random state.


In [ ]:
import umap
from umap.umap_ import nearest_neighbors
import hdbscan

unique_residuals, inverse = np.unique(residuals, axis=0, return_inverse=True)
np.save(CONTINUOUS_WORK / 'orthogonal_inverse.npy', inverse)

neighbors = nearest_neighbors(
    unique_residuals, n_neighbors=N_NEIGHBORS, metric='cosine',
    metric_kwds=None, angular=False, random_state=np.random.RandomState(SEED),
)
mapper = umap.UMAP(precomputed_knn=neighbors, n_neighbors=N_NEIGHBORS,
                   n_components=N_COMPONENTS, random_state=SEED)
orthogonal_coordinates = mapper.fit_transform(unique_residuals)
clusterer = hdbscan.HDBSCAN(min_samples=MIN_SAMPLES,
                          min_cluster_size=MIN_CLUSTER_SIZE)
unique_labels = clusterer.fit_predict(orthogonal_coordinates)

partition = pd.DataFrame(orthogonal_coordinates, columns=[f'd{i}' for i in range(N_COMPONENTS)])
partition['clus'] = unique_labels
partition.to_parquet(CONTINUOUS_WORK / 'orthogonal_partition.parquet', index=False)
pair_labels = unique_labels[inverse]
pairs['clus'] = pair_labels
pairs.to_parquet(CONTINUOUS_WORK / 'orthogonal_pair_labels.parquet', index=False)
pd.Series({'distinct_residuals': len(unique_residuals),
           'clusters': len(np.unique(unique_labels[unique_labels >= 0])),
           'noise_pair_share': np.mean(pair_labels == -1)})


### Centroids in the 50-dimensional CPC2vec space

Average the orthogonal residuals of all matched-pair rows assigned to each cluster other than the HDBSCAN noise group. Use the inverse mapping to weight duplicate residuals by their pair multiplicity.


In [ ]:
cluster_ids = np.unique(pair_labels[pair_labels >= 0])
centroids = np.vstack([residuals[pair_labels == label].mean(axis=0)
                       for label in cluster_ids])
centroids = centroids / np.linalg.norm(centroids, axis=1, keepdims=True)
centroid_table = pd.DataFrame(centroids, columns=[f'v{i}' for i in range(centroids.shape[1])])
centroid_table.insert(0, 'clus', cluster_ids)
centroid_table.to_parquet(CONTINUOUS_WORK / 'orthogonal_centroids.parquet', index=False)
centroid_table.head()


### Positive centroid–CPC profiles

Compare each normalized centroid with every code in the CPC2vec model and save positive cosine similarities. The complete positive profile supplies the RCA denominators in the next section.


In [ ]:
model_codes = np.asarray(cpc2vec.index_to_key)
code_vectors = cpc2vec.get_normed_vectors()
profiles = []
for label, center in zip(cluster_ids, centroids):
    similarity = code_vectors @ center
    positive = similarity > 0
    profiles.append(pd.DataFrame({
        'clus': label, 'cpc': model_codes[positive],
        'vecdiffcenter': similarity[positive],
    }))
continuous = pd.concat(profiles, ignore_index=True)
continuous['cpc'] = continuous.cpc.str.replace(' ', '', regex=False)
continuous.to_parquet(CONTINUOUS_PROFILE, index=False)
continuous.head()


## Continuous memberships and weighted correspondence

Calculate RCA across the entire positive centroid/code profile before applying RCA > 2 and intersecting with codes assigned to discrete GBBs. A CPC may belong to several continuous blocks.

In [ ]:
def continuous_correspondence(discrete, continuous):
    con = continuous.loc[continuous.vecdiffcenter > 0].copy()
    # The denominator is the entire positive continuous profile, before intersection.
    con = rca(con, "clus", "cpc", "vecdiffcenter")
    con = con.loc[con.rca > 2].rename(columns={"clus": "clus_alternative"})
    dis = nonnoise(discrete)[["cpc", "clus", "mean"]].rename(columns={"clus": "clus_reference"})
    shared = con.merge(dis, on="cpc")
    shared["score"] = shared.vecdiffcenter * shared["mean"]
    cells = rca(shared, "clus_reference", "clus_alternative", "score")
    return shared, cells


## Compare continuous and discrete GBBs

In [ ]:
discrete = pd.read_parquet(BASELINE / "clusangle_outlier_hdbscan2.parquet")
continuous = pd.read_parquet(CONTINUOUS_PROFILE, columns=["clus", "cpc", "vecdiffcenter"])
shared, cells = continuous_correspondence(discrete, continuous)
cells.to_csv(out / "continuous_correspondence.csv", index=False)
shared.to_parquet(out / "continuous_memberships.parquet", index=False)
result = dict(comparison="continuous", shared_codes=shared.cpc.nunique(),
              memberships=len(shared), **correspondence_metrics(cells))
pd.DataFrame([result]).to_csv(out / "continuous_nmi.csv", index=False)
print(result)
